[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/06_TF-IDF.ipynb)


In [ ]:
%pip install pandas scikit-learn

# TF-IDF: representar texto como vectores

En notebooks anteriores vimos cómo convertir texto en números: caracteres, índices de palabras, subpalabras. Pero todos esos métodos tienen un problema en común:

> **Para la máquina, `"no"` y `"sesión"` valen exactamente lo mismo si aparecen con la misma frecuencia.**

TF-IDF (*Term Frequency – Inverse Document Frequency*) resuelve eso. Es una técnica clásica de NLP que asigna un **peso** a cada palabra según dos criterios:

- **TF**: qué tan frecuente es esa palabra *dentro de un documento*
- **IDF**: qué tan rara es esa palabra *en el conjunto de todos los documentos*

El resultado: palabras específicas y poco comunes reciben pesos altos. Palabras genéricas que aparecen en todos los mensajes reciben pesos cercanos a cero.

---

## El problema: tickets de soporte de una app financiera

Imagina que eres parte del equipo de datos de un banco digital. El área de soporte recibe cientos de mensajes al día de usuarios con problemas.

Queremos saber **a qué categoría pertenece cada mensaje** para enrutarlo automáticamente al equipo correcto — sin leerlos uno a uno.

In [ ]:
# Nuestro corpus: tres mensajes de soporte
corpus = [
    "no puedo iniciar sesión",
    "no puedo hacer la transferencia",
    "el pago no se procesa"
]

for i, doc in enumerate(corpus, 1):
    print(f"Mensaje {i}: \"{doc}\"")

Para nosotros es evidente: los dos primeros son problemas de acceso y transacciones (el usuario no puede hacer algo), el tercero es un problema en el procesamiento de un pago.

Pero para la máquina son tres cadenas de texto sin relación. TF-IDF las convertirá en vectores numéricos que reflejen esa diferencia.

---

## Terminología

Antes de calcular, acordemos los términos que usaremos:

| Término | Definición | En este ejemplo |
|---|---|---|
| **Documento** | Cada texto individual | Cada mensaje de soporte |
| **Corpus** | El conjunto de todos los documentos | Los 3 mensajes |
| **Vocabulario** | Todas las palabras únicas del corpus, sin repetir | Las palabras sin función gramatical |
| **Stop word** | Palabra tan común que no aporta información útil | `"la"`, `"el"`, `"se"` |

In [ ]:
import re

# Excluimos palabras puramente gramaticales
STOP_WORDS = {"la", "el", "se"}

def tokenizar(texto):
    """Convierte un texto en lista de tokens en minúsculas, sin puntuación."""
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúüñ ]", "", texto)
    return [w for w in texto.split() if w not in STOP_WORDS]

# Vocabulario: todas las palabras únicas del corpus
vocabulario = sorted(set(
    token
    for doc in corpus
    for token in tokenizar(doc)
))

print("Vocabulario del corpus:\n")
for i, palabra in enumerate(vocabulario):
    print(f"  [{i:>2}]  {palabra}")
print(f"\nTotal: {len(vocabulario)} términos únicos")

Nota que `"no"` **sí** quedó en el vocabulario. Es una palabra importante semánticamente, pero vamos a ver qué le pasa cuando calculemos el IDF.

---

## Paso 1 — TF: Term Frequency

La **frecuencia del término** mide qué tan relevante es una palabra *dentro de un solo documento*.

```
TF(término, documento) = veces que aparece el término en el documento
                         ─────────────────────────────────────────────
                              total de términos en el documento
```

Se calcula **por documento por separado**. Entre más veces aparezca una palabra en un documento, mayor será su TF en ese documento.

In [ ]:
import math
import pandas as pd

def calcular_tf(documento, vocabulario):
    """TF de cada término del vocabulario en un documento."""
    tokens = tokenizar(documento)
    n = len(tokens)
    return {t: tokens.count(t) / n for t in vocabulario}

# Calcular TF para los tres mensajes
tfs = [calcular_tf(doc, vocabulario) for doc in corpus]

# Mostrar como tabla
df_tf = pd.DataFrame(
    tfs,
    index=[f"Msg {i+1}" for i in range(len(corpus))]
).round(3)

print("Matriz TF (frecuencia de cada término en cada mensaje):\n")
print(df_tf.to_string())

print("\nMensajes de referencia:")
for i, doc in enumerate(corpus, 1):
    print(f"  Msg {i}: {doc}")

Observa:

- `"no"` tiene TF > 0 en los **tres** mensajes — aparece en todos, pero no distingue uno del otro
- `"puedo"` tiene TF > 0 en los mensajes 1 y 2
- `"sesión"`, `"transferencia"`, `"pago"`, `"procesa"` tienen TF > 0 en **un solo** mensaje cada uno

El problema: aún no distinguimos entre `"no"` (muy común, poco informativa) y `"sesión"` (específica del Msg 1). Ambas pueden tener TF parecido. Para eso necesitamos el IDF.

---

## Paso 2 — IDF: Inverse Document Frequency

El **IDF** mide qué tan rara es una palabra *en todo el corpus*.

```
IDF(término) = ln( número total de documentos N )
                   ────────────────────────────────
                   número de docs que contienen el término
```

> Entre más documentos contengan la palabra, menor es su IDF.  
> Una palabra que aparece en **todos** los documentos tendrá IDF = ln(N/N) = ln(1) = **0**.

In [ ]:
def calcular_idf(corpus, vocabulario):
    """IDF de cada término del vocabulario en el corpus."""
    N = len(corpus)
    idfs = {}
    for t in vocabulario:
        df = sum(1 for doc in corpus if t in tokenizar(doc))
        idfs[t] = math.log(N / df)
    return idfs

idfs = calcular_idf(corpus, vocabulario)

print(f"N = {len(corpus)} documentos en el corpus\n")
print(f"  {'Término':<16} {'Aparece en':<20} {'IDF = ln(N/df)'}")
print("  " + "-" * 54)
for termino, valor in idfs.items():
    df = sum(1 for doc in corpus if termino in tokenizar(doc))
    print(f"  {termino:<16} {df}/{len(corpus)} documentos         ln({len(corpus)}/{df}) = {valor:.4f}")

El resultado clave:

- `"no"` aparece en los 3 mensajes → IDF = ln(3/3) = **0.0** — la máquina la ignorará por completo, aunque para nosotros parezca importante
- `"puedo"` aparece en 2 mensajes → IDF = ln(3/2) ≈ **0.405** — algo relevante, pero no decisivo
- `"sesión"`, `"transferencia"`, `"pago"`, `"procesa"`, etc. aparecen en solo 1 mensaje → IDF = ln(3/1) ≈ **1.099** — las más informativas

Esto es fundamental: **TF-IDF no necesita una lista de stop words exhaustiva**. Las palabras que aparecen en todos los documentos se anulan solas.

---

## Paso 3 — TF-IDF

Ahora combinamos ambas métricas multiplicando término a término:

```
TF-IDF(término, documento) = TF(término, documento) × IDF(término)
```

Palabras frecuentes *en el mensaje* pero raras *en el corpus* → peso alto.  
Palabras que aparecen en todos los mensajes → peso cero.

In [ ]:
def calcular_tfidf(tf_doc, idfs, vocabulario):
    """Vector TF-IDF de un documento."""
    return {t: tf_doc[t] * idfs[t] for t in vocabulario}

# Calcular TF-IDF para los tres mensajes
vectores = [calcular_tfidf(tf_doc, idfs, vocabulario) for tf_doc in tfs]

df_tfidf = pd.DataFrame(
    vectores,
    index=[f"Msg {i+1}" for i in range(len(corpus))]
).round(4)

print("Matriz TF-IDF:\n")
print(df_tfidf.to_string())

print("\nMensajes de referencia:")
for i, doc in enumerate(corpus, 1):
    print(f"  Msg {i}: {doc}")

In [ ]:
# Peso de cada término por mensaje: los que importan y los que se ignoran
print("Términos por mensaje:\n")
for i, (doc, vec) in enumerate(zip(corpus, vectores), 1):
    con_peso = {t: round(vec[t], 4) for t in vocabulario if vec[t] > 0}
    sin_peso  = [t for t in vocabulario if vec[t] == 0]

    con_peso = dict(sorted(con_peso.items(), key=lambda x: x[1], reverse=True))

    print(f"Msg {i}: \"{doc}\"")
    print(f"  Importan  →  {con_peso}")
    print(f"  Peso = 0  →  {sin_peso}")
    print()

---

## ¿Qué nos dicen estos vectores?

- **Msg 1** — `"no puedo iniciar sesión"`: las palabras con mayor peso son `iniciar` y `sesión`. Son las que identifican este mensaje como un problema de acceso.
- **Msg 2** — `"no puedo hacer la transferencia"`: `hacer` y `transferencia` tienen el mayor peso. `puedo` tiene algo de peso porque comparte ese término con el Msg 1.
- **Msg 3** — `"el pago no se procesa"`: `pago` y `procesa` tienen el mayor peso. Son las palabras únicas de este mensaje.
- En los tres mensajes, `"no"` tiene peso **0** — aunque parece importante, como aparece en todos los mensajes no ayuda a diferenciarlos.

Este último punto es la gran ventaja de TF-IDF: **no necesitas saber de antemano qué palabras son irrelevantes**. El método las descarta solo.

Estos vectores, todos del mismo tamaño, pueden usarse directamente como entrada para:

| Aplicación | Cómo se usa |
|---|---|
| **Clasificación de tickets** | Entrenar un modelo para categorizar: acceso / transacción / pago |
| **Agrupamiento** | Juntar automáticamente mensajes similares aunque estén redactados distinto |
| **Búsqueda** | Comparar un mensaje nuevo con los casos resueltos más parecidos |
| **Detección de tópicos** | Identificar los temas más frecuentes en miles de tickets |

---

## En producción: sklearn

Para escala real, usamos `TfidfVectorizer` de scikit-learn. Hace exactamente lo mismo pero de forma eficiente y con algunas variaciones en la fórmula.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer()
X = vectorizador.fit_transform(corpus)

df_sklearn = pd.DataFrame(
    X.toarray(),
    columns=vectorizador.get_feature_names_out(),
    index=[f"Msg {i+1}" for i in range(len(corpus))]
).round(4)

print("TF-IDF con sklearn:\n")
print(df_sklearn.to_string())

> **Nota**: sklearn usa una variante suavizada de la fórmula (`idf = ln((1+N)/(1+df)) + 1`) y normaliza los vectores para que tengan longitud 1. Por eso los valores son distintos a los que calculamos a mano, pero el principio es exactamente el mismo: palabras raras en el corpus y frecuentes en el documento tienen el mayor peso.

---

## Resumen

| Componente | Qué mide | Efecto |
|---|---|---|
| **TF** | Frecuencia de la palabra en el documento | Palabras frecuentes en ese mensaje → peso alto |
| **IDF** | Rareza de la palabra en el corpus | Palabras raras en el corpus → peso alto |
| **TF-IDF** | Producto de los dos | Palabras frecuentes en el mensaje *y* raras en el corpus tienen el mayor peso |

La idea central: **las palabras más informativas son las que distinguen un mensaje de los demás**, no las que aparecen en todos.

En nuestro ejemplo, `"no"` está en los tres mensajes → peso cero. `"sesión"`, `"transferencia"`, `"procesa"` están en uno solo → peso máximo. El sistema aprendió a ignorar el ruido sin que nadie se lo dijera explícitamente.